<a href="https://colab.research.google.com/github/zeynepuygrr/softito_hw/blob/main/24_09_2026odev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bölüm 1 — Keras ile ANN
**Veri seti:** Pima Indians Diabetes (768 hasta, 8 özellik, ikili hedef: diyabet var/yok)


In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Pima Indians Diabetes veri seti: 8 özellik + 1 ikili hedef (outcome)
cols = ["pregnancies", "glucose", "blood_pressure", "skin_thickness",
        "insulin", "bmi", "diabetes_pedigree", "age", "outcome"]
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
df = pd.read_csv(url, header=None, names=cols)

X = df.drop(columns=["outcome"]).values
y = df["outcome"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
model.summary()

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=50,
                     batch_size=16, verbose=2, callbacks=[early_stopping])

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test kaybı: {test_loss:.4f} | Test doğruluğu: {test_acc:.4f}")

# Bölüm 2 — PyTorch ile ANN
Aynı Pima veri setini (yukarıdaki `X_train`, `X_test`, `y_train`, `y_test`), bu kez PyTorch ile elle kurulan bir modelle eğitiyoruz.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [ ]:
full_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_ds, val_ds = random_split(full_ds, [int(0.8*len(full_ds)), len(full_ds)-int(0.8*len(full_ds))])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

In [ ]:
class ANN(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_features, 16),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(16, 1),
        nn.Dropout(0.2),
    )

  def forward(self, x):
    return self.net(x)

model = ANN(X_train.shape[1]).to(device)
print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()  # sigmoid + binary cross entropy
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def accuracy(logits, targets):
  preds = (torch.sigmoid(logits) > 0.5).float()
  return (preds == targets).float().mean()

In [ ]:
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
  model.train()
  train_loss = 0
  train_acc = 0
  for batch_idx, (data, targets) in enumerate(train_loader):
    data, targets = data.to(device), targets.to(device)
    optimizer.zero_grad()
    logits = model(data)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()*data.size(0)
    train_acc += accuracy(logits, targets)

In [ ]:
model.eval()
val_loss, val_correct = 0.0, 0.0
with torch.no_grad():
  for data, targets in val_loader:
    data, targets = data.to(device), targets.to(device)
    logits = model(data)
    val_loss += criterion(logits, targets).item()*data.size(0)
    val_correct += accuracy(logits, targets)*data.size(0)
val_loss /= len(val_loader.dataset)
val_acc = val_correct/len(val_loader.dataset)
print(f"Epoch: {epoch}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
  test_logits = model(X_test_tensor.to(device))
  test_loss = criterion(test_logits, y_test_tensor.to(device))
  test_acc = accuracy(test_logits, y_test_tensor.to(device))
print(f"\n Test kaybı: {test_loss:.4f} | Test doğruluğu {test_acc:.4f}")

# Bölüm 3 — Zaman Serisi için LSTM/GRU
**Veri seti:** Airline Passengers (1949-1960 arası aylık yolcu sayıları, 144 gözlem)

In [ ]:
url2 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df_air = pd.read_csv(url2)
series = df_air["Passengers"].values.astype("float32")

# Gerçek veri 100-600 aralığında olduğu için (sentetik sinüs dalgasının aksine)
# LSTM'in daha iyi öğrenmesi için ölçekliyoruz.
series_scaler = StandardScaler()
series_scaled = series_scaler.fit_transform(series.reshape(-1, 1)).flatten()

In [ ]:
WINDOW = 12  # 12 aylık pencere (yıllık mevsimsellik için mantıklı)

def make_windows(data, window):
  X, y = [], []
  for i in range(len(data)-window):
    X.append(data[i:i+window])
    y.append(data[i+window])
  return np.array(X), np.array(y)

X, y = make_windows(series_scaled, WINDOW)
# LSTM 3 boyutlu girdi bekler: (örnek_sayısı, zaman_adımı, özellik_sayısı)
X = X.reshape((X.shape[0], X.shape[1], 1))

In [ ]:
split = int(0.8*len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

In [ ]:
def build_model(cell="lstm", units=32):
  rnn_layer = layers.LSTM(units) if cell == "lstm" else layers.GRU(units)
  model = keras.Sequential([
      layers.Input(shape=(X_train.shape[1], 1)),
      rnn_layer,
      layers.Dense(1)
  ])
  model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
  return model

In [ ]:
results = {}

for cell in ["lstm", "gru"]:
  model = build_model(cell)
  model.summary()
  model.fit(
      X_train,
      y_train,
      epochs=20,
      batch_size=8,
      validation_split=0.1,
      verbose=2,
  )

In [ ]:
mse, mae = model.evaluate(X_test, y_test)
print("MSE", mse)
print("MAE", mae)

# Bölüm 4 — Metin (Duygu) Sınıflandırması için Embedding + LSTM
**Veri seti:** Twitter duygu veri seti (4000 tweet, ikili etiket: 0 = olumsuz, 1 = olumlu)

Hocanın IMDB veri setinin yerine bunu kullanıyoruz. IMDB, Keras içinde zaten sayısal dizilere çevrilmiş geliyordu; bizim verimiz ham metin olduğu için önce kendi küçük sözlüğümüzü (word_index) kurmamız gerekiyor — akışın geri kalanı (Embedding → LSTM → Dropout → Dense) birebir aynı.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 40      # tweetler IMDB yorumlarından çok daha kısa olduğu için düşürüldü
EMBED_DIM = 128
LSTM_UNITS = 64

In [ ]:
import re
from collections import Counter

url3 = "https://raw.githubusercontent.com/laxmimerit/twitter-data/master/twitter4000.csv"
df_tw = pd.read_csv(url3)
df_tw = df_tw.rename(columns={"twitts": "text", "sentiment": "label"})

def clean(text):
    words = re.sub(r"[^a-z0-9' ]", " ", str(text).lower()).split()
    return words

texts = df_tw["text"].apply(clean)
labels = df_tw["label"].values

print(f"Toplam örnek: {len(texts)}")
print("Örnek yorum:", " ".join(texts.iloc[0]))
print("Etiket:", "olumlu" if labels[0] == 1 else "olumsuz")

In [ ]:
# IMDB'de hazır gelen word_index'in yerine kendi sözlüğümüzü kelime sıklığına göre kuruyoruz.
# 0 = <PAD>, 1 = <START>, 2 = <OOV>, gerçek kelimeler 3'ten başlıyor (hocanınki ile aynı kural).
counter = Counter(w for words in texts for w in words)
most_common = [w for w, _ in counter.most_common(VOCAB_SIZE)]
word_index = {w: i for i, w in enumerate(most_common)}
index_word = {i+3: w for w, i in word_index.items()}
index_word.update({0: "<PAD>", 1: "<START>", 2: "<OOV>"})

def text_to_ids(words):
    ids = [1]  # <START>
    for w in words:
        idx = word_index.get(w)
        ids.append(idx + 3 if idx is not None and idx + 3 < VOCAB_SIZE else 2)
    return ids

sequences = [text_to_ids(w) for w in texts]

print("\n Örnek yorum (id):", "".join(index_word.get(i, "?") + " " for i in sequences[0][:15]), "....")

In [ ]:
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    sequences, labels, test_size=0.2, random_state=SEED, stratify=labels
)

X_train_text = keras.utils.pad_sequences(X_train_text, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_text = keras.utils.pad_sequences(X_test_text, maxlen=MAX_LEN, padding="post", truncating="post")

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, mask_zero=True),
    layers.LSTM(LSTM_UNITS),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(
    X_train_text, y_train_text,
    validation_split=0.1,
    epochs=15,
    batch_size=32,
    verbose=2,
    callbacks=[early_stopping],
)

test_loss, test_acc = model.evaluate(X_test_text, y_test_text, verbose=0)
print(f"Test kaybı: {test_loss:.4f} | Test doğruluğu: {test_acc:.4f}")

In [ ]:
def encode_text(text):
    """Ham İngilizce metni modelin beklediği indeks dizisine çevirir."""
    words = re.sub(r"[^a-z0-9' ]", " ", text.lower()).split()
    ids = [1]  # <START>
    for w in words:
        idx = word_index.get(w)
        ids.append(idx + 3 if idx is not None and idx + 3 < VOCAB_SIZE else 2)
    return keras.utils.pad_sequences([ids], maxlen=MAX_LEN, padding="post", truncating="post")

examples = [
    "This movie was absolutely wonderful, the acting was brilliant and I loved every minute.",
    "What a waste of time. The plot was boring and the characters were terrible.",
    "It was okay, not great but not bad either.",
]

print("\n--- Yeni yorumlar üzerinde tahmin ---")
for text in examples:
    prob = float(model.predict(encode_text(text), verbose=0)[0, 0])
    label = "OLUMLU" if prob > 0.5 else "OLUMSUZ"
    print(f"[{label} | {prob:.3f}] {text}")